In [1]:
pip install deepface openpyxl

Defaulting to user installation because normal site-packages is not writeable
  Using cached protobuf-7.35.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
Using cached protobuf-7.35.1-cp310-abi3-win_amd64.whl (439 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.9
    Uninstalling protobuf-4.25.9:
      Successfully uninstalled protobuf-4.25.9
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.72.0 requires protobuf!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
mediapipe 0.10.14 requires protobuf<5,>=4.25.3, but you have protobuf 7.35.1 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 7.35.1 which is incompatible.
streamlit 1.35.0 requires numpy<2,>=1.19.3, but you have numpy 2.5.0 which is incompatible.
streamlit 1.35.0 requires protobuf<5,>=3.20, but you have protobuf 7.35.1 w

In [2]:
import cv2
import pickle
import pandas as pd
import os

from datetime import datetime, time
from deepface import DeepFace

In [4]:
start_time = time(1,45)
end_time = time(2,15)

current_time = datetime.now().time()

if not(start_time <= current_time <= end_time):
    print("Attendance Closed")
    raise SystemExit

In [5]:
recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.read("trainer/trainer.yml")

with open("trainer/labels.pickle","rb") as f:
    labels = pickle.load(f)

face_detector = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")

In [6]:
students = os.listdir("dataset")

attendance = []

present_students = set()

In [7]:
cam = cv2.VideoCapture(0)

print("Attendance Started")

while True:

    ret, frame = cam.read()

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_detector.detectMultiScale(gray,1.3,5)

    for (x,y,w,h) in faces:

        face = gray[y:y+h,x:x+w]

        label, confidence = recognizer.predict(face)

        if confidence < 80:

            name = labels[label]

            if name not in present_students:

                emotion = DeepFace.analyze(
                    frame[y:y+h,x:x+w],
                    actions=['emotion'],
                    enforce_detection=False
                )

                emotion_name = emotion[0]['dominant_emotion']

                attendance.append({
                    "Name":name,
                    "Status":"Present",
                    "Emotion":emotion_name,
                    "Time":datetime.now().strftime("%H:%M:%S")
                })

                present_students.add(name)

                print(name,"Marked Present")

            cv2.putText(frame,name,(x,y-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8,(0,255,0),2)

        else:

            cv2.putText(frame,"Unknown",(x,y-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8,(0,0,255),2)

        cv2.rectangle(frame,(x,y),(x+w,y+h),(255,0,0),2)

    cv2.imshow("Attendance System",frame)

    if cv2.waitKey(1)==27:
        break

Attendance Started
Rubina Marked Present
Soheb Marked Present


In [8]:
for student in students:

    if student not in present_students:

        attendance.append({

            "Name":student,

            "Status":"Absent",

            "Emotion":"-",

            "Time":"-"

        })

In [9]:
df = pd.DataFrame(attendance)

df.to_csv("attendance.csv",index=False)

print(df)

print("Attendance Saved Successfully")

     Name   Status  Emotion      Time
0  Rubina  Present  neutral  01:59:49
1   Soheb  Present  neutral  02:00:00
Attendance Saved Successfully


In [10]:
cam.release()

cv2.destroyAllWindows()